# SI Figure S5: per-solvent PCM benefit vs bulk dielectric constant and polarizability

Per-solvent PCM benefit (% test-RMSE reduction) vs bulk dielectric constant and polarizability, for
the MagNET-Zero reference method per nucleus (WP04 ¹H, wB97X-D ¹³C). Panels A (¹H) and B (¹³C)
show all solvents; C repeats ¹H excluding aromatics and trifluoroethanol.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

import delta22
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# the published panels use 250 seeded train/test splits
N_SPLITS = 250

In [ ]:
dft = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
solutes = sorted(dft["solute"].unique())
benefit = {}
for nucleus, label in [("H", "1H"), ("C", "13C")]:
    method = delta22.MAGNET_PCM_OUTPUT_METHODS[nucleus]
    benefit[nucleus] = delta22.pcm_benefit_per_solvent(dft, method, "pcSseg2", "aimnet2",
                                                       delta22.DESMOND_SOLVENTS, n_splits=N_SPLITS,
                                                       solutes=solutes, nucleus=nucleus)
    print(f"{label} ({method}):"); print(benefit[nucleus].round(1).to_string())

## S5A (1H, all solvents), S5B (13C, all solvents), and S5C (1H, excluding aromatics and trifluoroethanol)

In [ ]:
# panels A (1H) and B (13C): all solvents
for nucleus, letter, nuc_label in [("H", "A", "H"), ("C", "B", "C")]:
    delta22_plots.plot_pcm_benefit_vs_properties(
        benefit[nucleus], delta22.SOLVENT_DIELECTRIC, delta22.SOLVENT_POLARIZABILITY, nuc_label,
        save_path=figure_path(f"si_figure_s05{letter}_all.png"))
# panel C is 1H only, excluding the aromatic solvents and trifluoroethanol (matches the published SI)
delta22_plots.plot_pcm_benefit_vs_properties(
    benefit["H"], delta22.SOLVENT_DIELECTRIC, delta22.SOLVENT_POLARIZABILITY, "H",
    exclude=["benzene", "toluene", "chlorobenzene", "trifluoroethanol"],
    title_extra="Excluding Aromatic Solvents and Trifluoroethanol",
    save_path=figure_path("si_figure_s05A_no_aromatics_tfe.png"))